# IMU Rep Counter Optimization - Colab Setup

This notebook imports the local Python modules so you can run your `train_and_eval()` logic inside a Colab environment.


In [ ]:
# Setup Environment and check PyTorch version
import torch
import sys

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")

# Check for GPU (CUDA for Colab NVIDIA GPUs, MPS for Apple Neural Engine)
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f"Active Device for Training: {device}")


Below is the Gru

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.nn.utils.rnn import pack_padded_sequence

class RNNPredictor(nn.Module):
    def __init__(self, input_size=10, hidden_size=32, num_classes=4, num_layers=1, batch_first=True):
        super().__init__()
        # input_size=10 because your IMU data has 10 columns (q0-g3, ax-az, gx-gz)
        # num_classes=4 because you are classifying 0, 1, 2, or 3 reps
        self.rnn = nn.GRU(input_size, hidden_size, num_layers=num_layers, batch_first=batch_first)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        # 1. pack_padded_sequence needs lengths to be on the CPU
        lengths = lengths.cpu()
        
        # 2. Pack the padded tensor so the GRU ignores the zeros
        # enforce_sorted=False is needed because we didn't sort the batch by length
        packed_x = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        
        # 3. Pass the packed sequence through the GRU
        _, h_n = self.rnn(packed_x)
        
        # 4. h_n shape is (num_layers, batch_size, hidden_size)
        # We grab the final hidden state of the very top layer
        last_hidden = h_n[-1]
        
        # 5. Pass it through the linear classification layer
        output = self.fc(last_hidden) # these are the raw logits for each class (not probabilities because we will use CrossEntropyLoss which applies softmax internally)
        
        return output

Below is the data prep

In [ ]:
import torch
import pandas as pd
import glob
import os
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

class IMURepDataset(Dataset):
    def __init__(self, data_dir, valid_subjects, valid_arms, valid_reps):
        """
        Args:
            data_dir (str): Path to the folder containing the CSVs.
            valid_subjects (list of str): Acceptable subjects.
            valid_arms (list of str): Acceptable arms.
            valid_reps (list of str): Acceptable reps.
        """
        file_paths = glob.glob(os.path.join(data_dir, "*.csv"))
        
        # Lists to hold the data in RAM
        self.sequences = []
        self.labels = []
        
        # Parse all files ONCE during initialization
        for file_path in file_paths:
            # 1. Parse Metadata
            meta_df = pd.read_csv(file_path, nrows=5, header=None, index_col=0)
            
            # Filter by acceptable subject, arm, and reps
            subject = str(meta_df.loc['subject', 1]).strip()
            arm = str(meta_df.loc['arm', 1]).strip()
            reps = str(meta_df.loc['reps', 1]).strip()
            
            if subject not in valid_subjects or arm not in valid_arms or reps not in valid_reps:
                continue

            label_val = int(meta_df.loc['label', 1])
            self.labels.append(torch.tensor(label_val, dtype=torch.long))
            
            # 2. Parse Actual Data
            data_df = pd.read_csv(file_path, skiprows=6)
            features = data_df[['q0', 'q1', 'q2', 'q3', 'ax', 'ay', 'az', 'gx', 'gy', 'gz']].values
            
            self.sequences.append(torch.tensor(features, dtype=torch.float32))

        # --- GLOBAL Z-SCORE NORMALIZATION ---
        # 1. Stack all sequences together into one giant tensor along the time dimension
        all_data = torch.cat(self.sequences, dim=0)

        # 2. Calculate the global mean and standard deviation for each of the 10 features
        # Keep dim=0 because we are flattening time and files, keeping only the 10 features
        self.mean = all_data.mean(dim=0)
        self.std = all_data.std(dim=0)

        # Prevent division by zero if a feature is completely constant
        self.std[self.std == 0] = 1.0

        # 3. Apply the global mean and std to standardize every individual sequence in the dataset
        for i in range(len(self.sequences)):
            self.sequences[i] = (self.sequences[i] - self.mean) / self.std
        input('pause')


    def __len__(self):
        # The length of the dataset is the number of samples (files) we have
        return len(self.sequences)

    def __getitem__(self, idx):
        # Returns the pre-parsed sequence and label for the given index when instance[] is called
        # Now this is lighting fast! Just returning from RAM.
        return self.sequences[idx], self.labels[idx]

def pad_collate(batch):
    """
    Custom collate function to handle variable-length IMU sequences.
    To be passed to the DataLoader: `DataLoader(..., collate_fn=pad_collate)`
    """
    # 'batch' is a list of tuples from __getitem__: [(seq1, label1), (seq2, label2), ...]
    sequences = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    labels = torch.stack(labels) # Convert list of labels to a tensor
    
    # Record the original lengths of each sequence before padding
    lengths = torch.tensor([len(seq) for seq in sequences], dtype=torch.long)
    
    # Pad the sequences. batch_first=True -> (batch_size, max_len, features)
    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=0.0)
    
    
    return sequences_padded, labels, lengths

Below is the training and evaluation sequence

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

# Import custom classes
from WorkUp_DATA import IMURepDataset, pad_collate
from WorkUp_GRU import RNNPredictor

def train_and_eval(lists):
    # 1. Setup Device (MPS for Mac, CUDA for Nvidia, otherwise CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # 2. Load the Dataset
    data_dir = os.path.join(os.path.dirname(__file__), "ESP32-C3 IMU", "Python Files", "recordings", "training1")
    print(f"Loading dataset from: {data_dir}")
    full_dataset = IMURepDataset(data_dir, *lists)
    print(f"Total labeled sequences loaded: {len(full_dataset)}")

    # 3. Split the Data (80% Training, 20% Testing)
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

    # 4. Create DataLoaders (using our custom collate_fn for the padding)
    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=pad_collate)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=pad_collate)

    # 5. Initialize the Model, Loss Function, and Optimizer
    model = RNNPredictor(input_size=10, hidden_size=32, num_classes=4).to(device)
    
    # CrossEntropyLoss expects discrete integer targets and raw unnormalized logits (this is AI jargon, but it works hehe)
    loss_fn = nn.CrossEntropyLoss() # aka criterion
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 40
    print("\nStarting Training...\n" + "-"*50)

    # 6. The Training Loop
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        correct_train = 0
        
        for sequences, labels, lengths in train_loader:
            sequences, labels = sequences.to(device), labels.to(device)

            # Reset gradients
            optimizer.zero_grad()
            
            # Forward pass (lengths stays on CPU as enforced inside the model)
            outputs = model(sequences, lengths)
            
            # Calculate loss
            loss = loss_fn(outputs, labels)
            
            # Backward pass & Optimize
            loss.backward()
            optimizer.step()

            # Track metrics
            total_train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1) # this is the classification prediction (0, 1, 2, or 3)
            correct_train += (predicted == labels).sum().item()

        # 7. Evaluation Phase (No backpropagation)
        model.eval()
        total_test_loss = 0
        correct_test = 0
        
        with torch.no_grad():
            for sequences, labels, lengths in test_loader:
                sequences, labels = sequences.to(device), labels.to(device)
                
                outputs = model(sequences, lengths)
                loss = loss_fn(outputs, labels)
                total_test_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1) # this is the classification prediction (0, 1, 2, or 3)
                correct_test += (predicted == labels).sum().item()

        # Print statistics
        train_acc = 100 * correct_train / train_size
        test_acc = 100 * correct_test / test_size
        avg_train_loss = total_train_loss / len(train_loader)
        avg_test_loss = total_test_loss / len(test_loader)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1:2d}/{epochs}] "
                  f"| Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:5.2f}% "
                  f"| Test Loss: {avg_test_loss:.4f} | Test Acc: {test_acc:5.2f}%")

if __name__ == "__main__":
    train_and_eval()

## Run Training Script

In [ ]:
# Run the training loop
A = [['Joshua Spiesman',
        'Ian Lake',
        'Tanner Gale',
        'Dean Black',
        'Brennan Johnson',
        'Reese Hammons',
        'Peter Cope'
        'Still with noise',
        'Two-hands and jerked',],['right', 'left', 'neither'],[0,1,2,3]]
B = [['Joshua Spiesman',
        'Ian Lake',
        'Tanner Gale',
        'Dean Black',
        'Brennan Johnson',
        'Reese Hammons',
        'Peter Cope'
        'Still with noise',
        'Two-hands and jerked',],['right', 'left', 'neither'],[0,1,2,3]]
C = [['Joshua Spiesman',
        'Ian Lake',
        'Tanner Gale',
        'Dean Black',
        'Brennan Johnson',
        'Reese Hammons',
        'Peter Cope'
        'Still with noise',
        'Two-hands and jerked',],['right', 'left', 'neither'],[0,1,2,3]]
D = [['Joshua Spiesman',
        'Ian Lake',
        'Tanner Gale',
        'Dean Black',
        'Brennan Johnson',
        'Reese Hammons',
        'Peter Cope'
        'Still with noise',
        'Two-hands and jerked',],['right', 'left', 'neither'],[0,1,2,3]]
E = [['Joshua Spiesman',
        'Ian Lake',
        'Tanner Gale',
        'Dean Black',
        'Brennan Johnson',
        'Reese Hammons',
        'Peter Cope'
        'Still with noise',
        'Two-hands and jerked',],['right', 'left', 'neither'],[0,1,2,3]]
F = [['Joshua Spiesman',
        'Ian Lake',
        'Tanner Gale',
        'Dean Black',
        'Brennan Johnson',
        'Reese Hammons',
        'Peter Cope'
        'Still with noise',
        'Two-hands and jerked',],['right', 'left', 'neither'],[0,1,2,3]]
lists = [A,B,C,D,E,F]
train_and_eval(lists[A])